Dada a base CIFAR10 que contém imagens organizadas em 10 classes, considere a criação de um classificador.

Estratégia a serem avaliadas:

4) uso de técnica de aumento de dados (data augmentation) durante o treinamento da rede.

Questões em aberto: 
a) Substituir o Flatten() por GlobalMaxPooling2D() impacta significativamente o resultado? 
b) Substituir o algoritmo otimizador (Adam()) pode melhor o resultado? 
c) Avaliar outras estratégias de data augmentation pode melhor o resultado?

In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import Sequential, layers, models
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

In [7]:
# 1. Importação direta da base CIFAR-10
# O Keras já separa automaticamente em treino e teste
(x_all, y_all), (x_test, y_test) = cifar10.load_data()

# 2. Normalização (crucial para o bom desempenho da CNN)
# As imagens estão em valores de 0-255, convertemos para 0-1
x_all = x_all.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 3. Divisão dos dados (Holdout)
# Agora estamos criando as variáveis 'trainX', 'valX', etc.
# Usamos 20% do conjunto de treino original para validação
trainX, valX, trainY, valY = train_test_split(
    x_all, y_all, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_all
)

# 4. Ajuste dos rótulos (One-Hot Encoding)
# A rede espera os rótulos no formato categórico (0-9 -> [0,0,1,0...])
trainY = tf.keras.utils.to_categorical(trainY, 10)
valY = tf.keras.utils.to_categorical(valY, 10)

In [8]:
# 1. Configuração do Data Augmentation
# Estas são estratégias comuns de aumento de dados para CIFAR-10/Imagens
datagen = ImageDataGenerator(
    rotation_range=20,      # Rotação aleatória
    width_shift_range=0.2,  # Deslocamento horizontal
    height_shift_range=0.2, # Deslocamento vertical
    horizontal_flip=True,   # Espelhamento horizontal
    zoom_range=0.2,         # Zoom aleatório
    fill_mode='nearest'     # Como preencher pixels novos
)

In [9]:
# 2. Carregar VGG16 e preparar o modelo
vgg_conv = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
vgg_conv.trainable = False 

In [10]:
model = Sequential([
    vgg_conv,
    layers.Flatten(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001) 

model.compile(loss='categorical_crossentropy', 
              optimizer=optimizer, 
              metrics=['accuracy'])

# 4. Treinamento
# O datagen.flow cria os lotes em tempo real durante o .fit()
train_generator = datagen.flow(trainX, trainY, batch_size=64)

history = model.fit(
    train_generator,
    validation_data=(valX, valY),
    epochs=50,
    verbose=1
)

Epoch 1/50


I0000 00:00:1782670332.757168   13677 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1782670333.639467   62770 service.cc:154] XLA service 0x78fa84512f60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782670333.639516   62770 service.cc:170]   StreamExecutor [0]: NVIDIA GeForce RTX 2060, Compute Capability 7.5 (Driver: 13.0.0[580.97.0]; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.2)
I0000 00:00:1782670333.710038   62770 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1782670334.196338   62770 cuda_dnn.cc:436] Loaded cuDNN version 92302


  7/625 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.1250 - loss: 2.3568

I0000 00:00:1782670336.558744   62770 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.3476 - loss: 1.8365

W0000 00:00:1782670354.973000   13677 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.
W0000 00:00:1782670355.083541   13677 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.


625/625 ━━━━━━━━━━━━━━━━━━━━ 27s 38ms/step - accuracy: 0.3476 - loss: 1.8365 - val_accuracy: 0.4790 - val_loss: 1.5238
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.4359 - loss: 1.6030 - val_accuracy: 0.5164 - val_loss: 1.4061
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 22s 36ms/step - accuracy: 0.4619 - loss: 1.5335 - val_accuracy: 0.5260 - val_loss: 1.3662
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.4771 - loss: 1.4939 - val_accuracy: 0.5440 - val_loss: 1.3230
Epoch 5/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.4870 - loss: 1.4635 - val_accuracy: 0.5512 - val_loss: 1.2939
Epoch 6/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.4922 - loss: 1.4420 - val_accuracy: 0.5534 - val_loss: 1.2819
Epoch 7/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.4996 - loss: 1.4238 - val_accuracy: 0.5556 - val_loss: 1.2723
Epoch 8/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.5062 - loss: 1.4121 - val_accurac

In [11]:
model = Sequential([
    vgg_conv,
    # Questão (a): GlobalMaxPooling2D vs Flatten
    layers.GlobalMaxPooling2D(), 
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

# 3. Questão (b): Otimizador
# Adam é excelente, mas o SGD com momentum é mais estável para fine-tuning refinado
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001) 

model.compile(loss='categorical_crossentropy', 
              optimizer=optimizer, 
              metrics=['accuracy'])

# 4. Treinamento
# O datagen.flow cria os lotes em tempo real durante o .fit()
train_generator = datagen.flow(trainX, trainY, batch_size=64)

history = model.fit(
    train_generator,
    validation_data=(valX, valY),
    epochs=50,
    verbose=1
)

Epoch 1/50
623/625 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3449 - loss: 1.8431

W0000 00:00:1782671317.296545   13677 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.
W0000 00:00:1782671317.389211   13677 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.


625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.3453 - loss: 1.8422 - val_accuracy: 0.4899 - val_loss: 1.5164
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.4373 - loss: 1.6076 - val_accuracy: 0.5143 - val_loss: 1.4170
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.4615 - loss: 1.5368 - val_accuracy: 0.5293 - val_loss: 1.3655
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.4737 - loss: 1.4978 - val_accuracy: 0.5383 - val_loss: 1.3307
Epoch 5/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.4872 - loss: 1.4653 - val_accuracy: 0.5368 - val_loss: 1.3246
Epoch 6/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.4943 - loss: 1.4491 - val_accuracy: 0.5501 - val_loss: 1.2890
Epoch 7/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.4980 - loss: 1.4238 - val_accuracy: 0.5554 - val_loss: 1.2736
Epoch 8/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.5082 - loss: 1.4088 - val_accurac

In [12]:
# 3. Arquitetura com FLATTEN
model_flatten = Sequential([
    vgg_conv,
    layers.Flatten(), # Mantido conforme solicitado
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

# 4. Questão (b): Otimizador SGD com momentum
# O SGD com momentum é frequentemente mais estável e generaliza melhor que o Adam em Fine-Tuning
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)

model_flatten.compile(loss='categorical_crossentropy', 
                      optimizer=optimizer, 
                      metrics=['accuracy'])

model_flatten.summary()

# 5. Treinamento
# Substitua 'trainX', 'trainY', 'valX', 'valY' pelas suas bases já divididas
train_generator = datagen.flow(trainX, trainY, batch_size=64)

history = model_flatten.fit(
    train_generator,
    validation_data=(valX, valY),
    epochs=50,
    verbose=1
)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 1, 1, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,250,250 (58.18 MB)

 Trainable params: 535,562 (2.04 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.2921 - loss: 1.9871

W0000 00:00:1782672256.359885   13677 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.


625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 30ms/step - accuracy: 0.2921 - loss: 1.9871 - val_accuracy: 0.4216 - val_loss: 1.7117
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.3779 - loss: 1.7653 - val_accuracy: 0.4600 - val_loss: 1.5856
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.4067 - loss: 1.6867 - val_accuracy: 0.4761 - val_loss: 1.5325
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.4211 - loss: 1.6418 - val_accuracy: 0.4941 - val_loss: 1.4728
Epoch 5/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.4347 - loss: 1.6081 - val_accuracy: 0.5047 - val_loss: 1.4426
Epoch 6/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.4426 - loss: 1.5830 - val_accuracy: 0.5113 - val_loss: 1.4271
Epoch 7/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.4496 - loss: 1.5632 - val_accuracy: 0.5167 - val_loss: 1.4007
Epoch 8/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.4539 - loss: 1.5499 - val_accurac

In [13]:
datagen_avancado = ImageDataGenerator(
    rotation_range=20,      # Rotação aleatória
    width_shift_range=0.2,  # Deslocamento horizontal
    height_shift_range=0.2, # Deslocamento vertical
    horizontal_flip=True,   # Espelhamento horizontal
    zoom_range=0.2,         # Zoom aleatório    
    # 1. Shear (Cisalhamento): Inclina a imagem, simulando uma mudança na perspectiva.
    shear_range=0.2,
    # 2. Brilho (Brightness): Varia a iluminação. Ajuda muito em cenários do mundo real.
    brightness_range=[0.8, 1.2],
    # 3. Channel Shift: Altera os valores dos canais de cor RGB. 
    # Isso torna a rede menos dependente de cores específicas para identificar o objeto.
    channel_shift_range=20.0,
    # 5. Preenchimento: Quando rotacionamos/deslocamos, sobram espaços vazios.
    fill_mode='reflect' # 'reflect' costuma gerar menos artefatos que o 'nearest'
)

In [14]:
model = Sequential([
    vgg_conv,
    layers.Flatten(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001) 

model.compile(loss='categorical_crossentropy', 
              optimizer=optimizer, 
              metrics=['accuracy'])

# 4. Treinamento
# O datagen.flow cria os lotes em tempo real durante o .fit()
train_generator = datagen.flow(trainX, trainY, batch_size=64)

history = model.fit(
    train_generator,
    validation_data=(valX, valY),
    epochs=50,
    verbose=1
)

Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.3496 - loss: 1.8382 - val_accuracy: 0.4923 - val_loss: 1.5033
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.4366 - loss: 1.6021 - val_accuracy: 0.5154 - val_loss: 1.4082
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.4593 - loss: 1.5382 - val_accuracy: 0.5277 - val_loss: 1.3650
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.4735 - loss: 1.4968 - val_accuracy: 0.5388 - val_loss: 1.3280
Epoch 5/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.4870 - loss: 1.4676 - val_accuracy: 0.5463 - val_loss: 1.3087
Epoch 6/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.4933 - loss: 1.4425 - val_accuracy: 0.5503 - val_loss: 1.2958
Epoch 7/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.4983 - loss: 1.4251 - val_accuracy: 0.5567 - val_loss: 1.2762
Epoch 8/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.5037 - loss: 1.4082 - 